# 每只股票的 Polymarket 敏感度 $b_i$

读取训练时导出的 `sensitivities.csv`，展示每只股票的五个 $b_i$，以及**每个 $b_i$ 数值最大的 10 只股票及其名称**。

| 参数 | CSV 列名 | 类别 |
| --- | --- | --- |
| $b_1$ | `direction` | 区间涨跌（日度与多日共用） |
| $b_2$ | `threshold` | 收盘超过门槛 |
| $b_3$ | `touch` | 期间触价（上触与下触共用） |
| $b_4$ | `range` | 结算价格区间 |
| $b_5$ | `record` | 历史新高 |

每个测试月的模型独立初始化和训练，因此以下排名始终针对**一个训练批次、一个测试月**。
参数初始值为 1，允许为负；这里只按原始数值降序排名，不取绝对值。
这些数值表示模型内部的相对敏感度，尺度还受共享编码器权重影响。

从项目根目录或 `explore` 目录运行均可。使用项目 `.venv` 的 Python 内核，依次运行所有单元格即可；默认无需联网。

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

# None 表示自动选择最新批次和该批次最新的已完成月份。
# 例如 RUN_ID = "20261022T110412673021"，TEST_MONTH = "2026-08"。
# 旧版直接存于 outputs/polymarket/YYYY-MM 的结果用 RUN_ID = "."。
RUN_ID = None
TEST_MONTH = None
TOP_N = 10

# 留空展示全部训练股票；可以按代码或名称筛选，例如 "600519" 或 "茅台"。
STOCK_QUERY = ""

root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in root_candidates
     if (path / "polymarket_ext" / "model.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从项目根目录或 explore 目录运行此 notebook。")

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "polymarket"
NAME_PATH = PROJECT_ROOT / "explore" / "stock_names.csv"
CATEGORIES = {
    "direction": ("b_1", "区间涨跌"),
    "threshold": ("b_2", "收盘超过门槛"),
    "touch": ("b_3", "期间触价"),
    "range": ("b_4", "结算价格区间"),
    "record": ("b_5", "历史新高"),
}
pd.set_option("display.max_columns", 20)

## 选择已训练的模型

下面列出有 `completed.json` 和参数文件的月份。默认选择名称最新的训练批次，再选择其中最新的已完成月份；可以修改上面的 `RUN_ID` 和 `TEST_MONTH` 后重新运行。

In [2]:
records = []
# 兼容 outputs/polymarket/<run_id>/<month>/ 和旧版 <month>/ 两种布局。
paths = set(OUTPUT_ROOT.glob("????-??/sensitivities.csv"))
paths.update(OUTPUT_ROOT.glob("*/????-??/sensitivities.csv"))
for path in sorted(paths):
    month_dir = path.parent
    if not (month_dir / "completed.json").is_file():
        continue
    run_path = month_dir / "run.json"
    run = json.loads(run_path.read_text(encoding="utf-8")) if run_path.exists() else {}
    records.append({
        "批次": month_dir.parent.relative_to(OUTPUT_ROOT).as_posix(),
        "测试月份": month_dir.name,
        "训练轮数": run.get("trained_epochs"),
        "训练起点": run.get("window", {}).get("train_start"),
        "训练终点": run.get("window", {}).get("train_end"),
        "参数文件": path,
    })

if not records:
    raise FileNotFoundError(f"{OUTPUT_ROOT} 下未找到已完成的月度 sensitivities.csv。")
catalog = pd.DataFrame(records).sort_values(["批次", "测试月份"]).reset_index(drop=True)
display(catalog.drop(columns="参数文件"))

run_ids = sorted(catalog["批次"].unique())
selected_run = RUN_ID if RUN_ID is not None else run_ids[-1]
available = catalog.loc[catalog["批次"].eq(selected_run)]
if available.empty:
    raise ValueError(f"不存在批次 {selected_run!r}；可选批次：{run_ids}")
selected_month = TEST_MONTH if TEST_MONTH is not None else available["测试月份"].max()
selected = available.loc[available["测试月份"].eq(selected_month)]
if selected.empty:
    raise ValueError(
        f"批次 {selected_run} 没有已完成的 {selected_month}；"
        f"可选月份：{available['测试月份'].tolist()}"
    )

selection = selected.iloc[0]
sensitivity_path = selection["参数文件"]
completed = json.loads(
    (sensitivity_path.parent / "completed.json").read_text(encoding="utf-8")
)
expected_hash = completed.get("artifacts", {}).get("sensitivities.csv")
if expected_hash and hashlib.sha256(sensitivity_path.read_bytes()).hexdigest() != expected_hash:
    raise ValueError("sensitivities.csv 与 completed.json 中的校验值不一致。")

coefficients = pd.read_csv(sensitivity_path, dtype={"instrument": str})
required = ["instrument", *CATEGORIES]
missing_columns = set(required) - set(coefficients.columns)
if missing_columns:
    raise ValueError(f"参数文件缺少列：{sorted(missing_columns)}")
coefficients = coefficients[required].copy()
if coefficients.empty or coefficients["instrument"].isna().any():
    raise ValueError("参数文件为空或存在缺失股票代码。")
if coefficients["instrument"].duplicated().any():
    raise ValueError("参数文件中同一股票存在多行。")
coefficients[list(CATEGORIES)] = coefficients[list(CATEGORIES)].apply(
    pd.to_numeric, errors="raise"
)
if not np.isfinite(coefficients[list(CATEGORIES)].to_numpy()).all():
    raise ValueError("训练参数中含 NaN 或无穷值，请检查该次训练输出。")

display(Markdown(
    f"**当前模型：** 批次 **{selected_run}**，测试月 **{selected_month}**，"
    f"训练 **{selection['训练轮数']}** 轮，"
    f"训练段 {selection['训练起点']} 至 {selection['训练终点']}。  \n"
    f"**参数来源：** {sensitivity_path.relative_to(PROJECT_ROOT)}  \n"
    f"**股票数量：** {len(coefficients)} 只。这是训练中出现过的历史股票全集，"
    "不限定为测试月的 300 只成分股；训练中未出现的股票使用的中性值 1 不另行补入。"
))

,批次,测试月份,训练轮数,训练起点,训练终点
0,20261022T110412673021,2026-04,10,2008-01-01,2026-03-27
1,20261022T110412673021,2026-05,10,2008-01-01,2026-04-28
2,20261022T110412673021,2026-06,10,2008-01-01,2026-05-27
3,20261022T110412673021,2026-07,10,2008-01-01,2026-06-26
4,20261022T110412673021,2026-08,10,2008-01-01,2026-07-29
5,20261022T110412673021,2026-09,10,2008-01-01,2026-08-27
6,20261022T114144584340,2026-04,20,2008-01-01,2026-03-27
7,20261022T114144584340,2026-05,20,2008-01-01,2026-04-28
8,20261022T114144584340,2026-06,20,2008-01-01,2026-05-27
9,20261022T114144584340,2026-07,20,2008-01-01,2026-06-26


**当前模型：** 批次 **20261022T114144584340**，测试月 **2026-09**，训练 **20** 轮，训练段 2008-01-01 至 2026-08-27。  
**参数来源：** outputs/polymarket/20261022T114144584340/2026-09/sensitivities.csv  
**股票数量：** 859 只。这是训练中出现过的历史股票全集，不限定为测试月的 300 只成分股；训练中未出现的股票使用的中性值 1 不另行补入。

## 股票名称

使用随 notebook 保存的 `explore/stock_names.csv`，名称来源是 BaoStock 的 `query_stock_basic()`，包含已退市股票。
名称是抓取时的展示快照，**不代表模型训练当时的历史简称**，也不参与参数计算和排名。代码保留 `SH/SZ` 前缀和前导零。

如以后新增训练股票未被该快照覆盖，表中会保留代码并明确标记名称缺失；可以补充此 CSV 后重新运行。

In [3]:
if not NAME_PATH.is_file():
    raise FileNotFoundError(f"缺少股票名称快照：{NAME_PATH}")
names = pd.read_csv(NAME_PATH, dtype=str)
if not {"instrument", "name"}.issubset(names.columns):
    raise ValueError("stock_names.csv 需要 instrument 和 name 列。")
if names["instrument"].duplicated().any():
    raise ValueError("名称快照中同一股票代码存在多行。")

stock_parameters = coefficients.merge(
    names[["instrument", "name"]], on="instrument", how="left", validate="one_to_one"
)
missing_names = stock_parameters["name"].isna() | stock_parameters["name"].eq("")
unmatched_names = stock_parameters.loc[missing_names, ["instrument"]]
stock_parameters.loc[missing_names, "name"] = "名称缺失"
stock_parameters = stock_parameters.sort_values("instrument").reset_index(drop=True)

print(f"名称匹配：{len(stock_parameters) - missing_names.sum()} / {len(stock_parameters)}")
if "fetched_at" in names:
    print("名称快照抓取时间：", ", ".join(sorted(names["fetched_at"].dropna().unique())))
if not unmatched_names.empty:
    display(unmatched_names.rename(columns={"instrument": "缺少名称的股票代码"}))

column_labels = {
    "instrument": "股票代码",
    "name": "股票名称",
    **{category: parameter for category, (parameter, _) in CATEGORIES.items()},
}

名称匹配：859 / 859
名称快照抓取时间： 2026-10-22T14:36:15+08:00


## 每只股票的五个 $b_i$

下表展示完整参数，保留 **10 位小数**，可在表格内滚动查看。设置开头的 `STOCK_QUERY` 可以按股票代码或名称筛选；此筛选只影响完整表，不影响后面的全体股票 Top 10。

In [4]:
visible = stock_parameters
if STOCK_QUERY.strip():
    query = STOCK_QUERY.strip()
    matches = (
        visible["instrument"].str.contains(query, case=False, regex=False)
        | visible["name"].str.contains(query, case=False, regex=False)
    )
    visible = visible.loc[matches]

all_stock_b = visible[["instrument", "name", *CATEGORIES]].rename(columns=column_labels)
print(f"展示 {len(all_stock_b)} / {len(stock_parameters)} 只股票")
display(HTML(
    '<div style="max-height:560px;overflow:auto;">'
    + all_stock_b.to_html(index=False, float_format=lambda x: f"{x:.10f}", escape=True)
    + "</div>"
))

展示 859 / 859 只股票


股票代码,股票名称,b_1,b_2,b_3,b_4,b_5
SH600000,浦发银行,1.0000010000,0.9999994000,1.0000012000,1.0000000000,1.0000012000
SH600001,邯郸钢铁,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000
SH600003,ST东北高,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000
SH600004,白云机场,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000
SH600005,武钢股份,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000
SH600006,东风股份,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000
SH600007,中国国贸,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000
SH600008,首创环保,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000
SH600009,上海机场,1.0000000000,1.0000012000,0.9999994600,0.9999998000,0.9999994000
SH600010,包钢股份,0.9999999400,0.9999994000,1.0000006000,0.9999996000,0.9999994000


## 参数分布与并列情况

同时检查参数距离初始值 1 有多远，以及每类最大值有多少只股票并列。参数非常接近 1 时，少量小数差异和股票代码排序都会影响表中的先后，不能据此推断稳定的经济敏感度。

In [5]:
summary_rows = []
for category, (parameter, label) in CATEGORIES.items():
    values = stock_parameters[category]
    summary_rows.append({
        "参数": parameter,
        "类别": label,
        "股票数": len(values),
        "不同取值数": values.nunique(),
        "最小值": values.min(),
        "中位数": values.median(),
        "最大值": values.max(),
        "最大绝对偏离1": (values - 1).abs().max(),
        "恰好等于1的股票数": values.eq(1).sum(),
        "最大值并列股票数": values.eq(values.max()).sum(),
    })
parameter_summary = pd.DataFrame(summary_rows)
display(HTML(parameter_summary.to_html(
    index=False, float_format=lambda x: f"{x:.10f}", escape=True
)))

参数,类别,股票数,不同取值数,最小值,中位数,最大值,最大绝对偏离1,恰好等于1的股票数,最大值并列股票数
b_1,区间涨跌,859,21,0.9999994000,1.0000000000,1.0000012000,0.0000012000,584,14
b_2,收盘超过门槛,859,3,0.9999994000,1.0000000000,1.0000012000,0.0000012000,540,178
b_3,期间触价,859,24,0.9999993000,1.0000000000,1.0000013000,0.0000013000,578,12
b_4,结算价格区间,859,21,0.9999994000,1.0000000000,1.0000012000,0.0000012000,599,20
b_5,历史新高,859,3,0.9999994000,1.0000000000,1.0000012000,0.0000012000,540,155


## 各类 $b_i$ 最大的 10 只股票

每类单独按 $b_i$ **原始数值从大到小**排列；数值相同时按股票代码升序排列，以便结果可复现。
“数值排名”对并列股票给出相同名次，“同值股票数”统计该模型所有股票中的并列数量。

若第 10 行仍有并列，表中只展示按代码排序后的 10 行，并报告达到该行数值的股票总数。
`top10_by_category` 保留五张结果表，方便继续分析。

In [6]:
if not isinstance(TOP_N, int) or TOP_N < 1:
    raise ValueError("TOP_N 必须是正整数。")

top10_by_category = {}
for category, (parameter, label) in CATEGORIES.items():
    ranked = stock_parameters[["instrument", "name", category]].copy()
    ranked["数值排名"] = ranked[category].rank(
        method="min", ascending=False
    ).astype(int)
    ranked["同值股票数"] = ranked[category].map(ranked[category].value_counts())
    ranked = ranked.sort_values(
        [category, "instrument"], ascending=[False, True]
    ).head(TOP_N)
    ranked["较1偏离"] = ranked[category] - 1
    top = ranked[
        ["数值排名", "instrument", "name", category, "较1偏离", "同值股票数"]
    ].rename(columns=column_labels).reset_index(drop=True)
    top10_by_category[category] = top

    display(Markdown(f"### {parameter}：{label}"))
    display(HTML(top.to_html(
        index=False, float_format=lambda x: f"{x:.10f}", escape=True
    )))
    cutoff = ranked[category].iloc[-1]
    values = stock_parameters[category]
    tied = int(values.eq(cutoff).sum())
    higher = int(values.gt(cutoff).sum())
    print(
        f"第 {len(top)} 行的 {parameter} = {cutoff:.10f}；"
        f"高于该值 {higher} 只，等于该值 {tied} 只。"
    )
    if higher + tied > len(top):
        print("该行存在未全部展示的并列股票，表中按股票代码升序截取。")

### b_1：区间涨跌

数值排名,股票代码,股票名称,b_1,较1偏离,同值股票数
1,SH600016,民生银行,1.0000012000,0.0000012000,14
1,SH600219,南山铝业,1.0000012000,0.0000012000,14
1,SH600489,中金黄金,1.0000012000,0.0000012000,14
1,SH600519,贵州茅台,1.0000012000,0.0000012000,14
1,SH601066,中信建投,1.0000012000,0.0000012000,14
1,SH601238,广汽集团,1.0000012000,0.0000012000,14
1,SH603260,合盛硅业,1.0000012000,0.0000012000,14
1,SH603986,兆易创新,1.0000012000,0.0000012000,14
1,SZ000786,北新建材,1.0000012000,0.0000012000,14
1,SZ000999,华润三九,1.0000012000,0.0000012000,14


第 10 行的 b_1 = 1.0000012000；高于该值 0 只，等于该值 14 只。
该行存在未全部展示的并列股票，表中按股票代码升序截取。


### b_2：收盘超过门槛

数值排名,股票代码,股票名称,b_2,较1偏离,同值股票数
1,SH600009,上海机场,1.0000012000,0.0000012000,178
1,SH600011,华能国际,1.0000012000,0.0000012000,178
1,SH600015,华夏银行,1.0000012000,0.0000012000,178
1,SH600016,民生银行,1.0000012000,0.0000012000,178
1,SH600018,上港集团,1.0000012000,0.0000012000,178
1,SH600019,宝钢股份,1.0000012000,0.0000012000,178
1,SH600025,华能水电,1.0000012000,0.0000012000,178
1,SH600027,华电国际,1.0000012000,0.0000012000,178
1,SH600036,招商银行,1.0000012000,0.0000012000,178
1,SH600039,四川路桥,1.0000012000,0.0000012000,178


...

数值排名,股票代码,股票名称,b_4,较1偏离,同值股票数
1,SH600061,国投资本,1.0000012000,0.0000012000,20
1,SH600196,复星医药,1.0000012000,0.0000012000,20
1,SH600221,海航控股,1.0000012000,0.0000012000,20
1,SH600438,通威股份,1.0000012000,0.0000012000,20
1,SH600547,山东黄金,1.0000012000,0.0000012000,20
1,SH601398,工商银行,1.0000012000,0.0000012000,20
1,SH601808,中海油服,1.0000012000,0.0000012000,20
1,SH688072,拓荆科技,1.0000012000,0.0000012000,20
1,SH688126,沪硅产业,1.0000012000,0.0000012000,20
1,SH688183,生益电子,1.0000012000,0.0000012000,20


第 10 行的 b_4 = 1.0000012000；高于该值 0 只，等于该值 20 只。
该行存在未全部展示的并列股票，表中按股票代码升序截取。


### b_5：历史新高

数值排名,股票代码,股票名称,b_5,较1偏离,同值股票数
1,SH600000,浦发银行,1.0000012000,0.0000012000,155
1,SH600011,华能国际,1.0000012000,0.0000012000,155
1,SH600018,上港集团,1.0000012000,0.0000012000,155
1,SH600019,宝钢股份,1.0000012000,0.0000012000,155
1,SH600029,南方航空,1.0000012000,0.0000012000,155
1,SH600085,同仁堂,1.0000012000,0.0000012000,155
1,SH600115,中国东航,1.0000012000,0.0000012000,155
1,SH600118,中国卫星,1.0000012000,0.0000012000,155
1,SH600150,中国船舶,1.0000012000,0.0000012000,155
1,SH600161,天坛生物,1.0000012000,0.0000012000,155


第 10 行的 b_5 = 1.0000012000；高于该值 0 只，等于该值 155 只。
该行存在未全部展示的并列股票，表中按股票代码升序截取。


## 继续分析

上面的对象可直接复用：

- `stock_parameters`：选定模型所有训练股票的原始参数和名称。
- `all_stock_b`：应用 `STOCK_QUERY` 后的完整展示表。
- `parameter_summary`：五类参数的分布统计。
- `top10_by_category["direction"]`：$b_1$ 的 Top 10；其他类别键见开头的对应表。

如需导出，可手动执行 `stock_parameters.to_csv("stock_parameters.csv", index=False, encoding="utf-8-sig")`。
修改批次或月份后，从头运行即可；不对不同月份的独立模型参数取平均。